# 15 — Tucker Decomposition + PCA + Robust Mahalanobis Clustering

Uses the Tucker-compressed patient representation from notebook 14 for clustering.

**Pipeline:**
1. Raw HOG tensor (416 × 8,100 × 5) → Tucker rank (416, 1000, 3)
2. Flatten core → (416, 3,000)
3. PCA 90% → **~57 components** (safely below 138 Mahalanobis limit)
4. Robust Mahalanobis distance matrix (416 × 416)
5. FasterPAM k-medoids for k=3 and k=4
6. Evaluate: spread · silhouette · ARI · NMI

Same evaluation structure as notebook 13 (two-stage PCA baseline).

## Install

In [1]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r   = subprocess.run(cmd, capture_output=True, text=True)
    tag = args[-1]
    if r.returncode == 0:
        print(f"  OK    {tag}")
    else:
        print(f"  FAIL  {tag}")
        print(r.stderr[-400:] if r.stderr else "(no stderr)")

for pkg in ["tensorly", "kmedoids", "robust-mixed-dist", "openpyxl"]:
    run_pip(pkg)
print("Done.")

  OK    tensorly
  OK    kmedoids
  OK    robust-mixed-dist
  OK    openpyxl
Done.


## Imports

In [2]:
import os
import time
import numpy as np
import pandas as pd
import tensorly as tl
from tensorly.decomposition import tucker
from sklearn.decomposition import PCA

import kmedoids
from robust_mixed_dist.mixed import robust_mahalanobis_dist_matrix, S_robust
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print("All imports successful.")

All imports successful.


## Step 1 — Load raw HOG features and CDR

In [3]:
NB_FOLDER = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NB_FOLDER, '..', '..'))
DATA_PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'notebooks', '01_data_pipeline')
OUTPUT_DIR = NB_FOLDER
NB_DIR = DATA_PIPELINE_DIR      # legacy alias: shared pipeline inputs (pca_*.csv, clinical xlsx)
NB_PADDED = OUTPUT_DIR          # legacy alias: this notebook's own outputs

HOG_LEN = 8_100

# ── Raw HOG features ──────────────────────────────────────────────────────────
print("Loading raw_hog_features_per_image.csv ...", end=" ", flush=True)
df_hog      = pd.read_csv(os.path.join(NB_DIR, "raw_hog_features_per_image.csv"))
patient_ids = df_hog["patient_id"].values
X_flat      = df_hog.drop(columns=["patient_id"]).to_numpy(dtype=np.float64)
print(f"done  {X_flat.shape}")

# ── Build 3D tensor (416, 8100, 5) ────────────────────────────────────────────
blocks    = [X_flat[:, i * HOG_LEN:(i + 1) * HOG_LEN] for i in range(5)]
tensor    = np.stack(blocks, axis=2)
tensor_tl = tl.tensor(tensor)
print(f"Tensor shape : {tensor.shape}")

# ── CDR ───────────────────────────────────────────────────────────────────────
df_clin = pd.read_excel(
    os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
)
if "ID" in df_clin.columns:
    df_clin = df_clin.rename(columns={"ID": "patient_id"})
df_clin["patient_id"] = df_clin["patient_id"].str.replace(r"_MR\d+$", "", regex=True)
df_clin = (
    df_clin
    .sort_values("CDR", na_position="last")
    .drop_duplicates(subset="patient_id", keep="first")
    .reset_index(drop=True)
)
cdr_series = df_clin.set_index("patient_id")["CDR"].reindex(patient_ids)
mask_cdr   = cdr_series.notnull().values
cdr_int    = (cdr_series.values[mask_cdr] * 2).astype(int)

print(f"Patients with CDR : {mask_cdr.sum()} / {len(patient_ids)}")

Loading raw_hog_features_per_image.csv ... done  (416, 40500)
Tensor shape : (416, 8100, 5)
Patients with CDR : 235 / 416


## Step 2 — Tucker decomposition

Rank `(416, 1000, 3)`: keep all patient rows, compress HOG to 1,000, keep 3 image-type directions.  
Flatten core → `(416, 3,000)` patient representation.

In [4]:
TUCKER_RANK = (416, 1000, 3)

print(f"Tucker decomposition rank={TUCKER_RANK} ...", end=" ", flush=True)
t0 = time.time()
core, factors = tucker(tensor_tl, rank=TUCKER_RANK, random_state=42)
print(f"done in {time.time()-t0:.1f}s")

core_np      = tl.to_numpy(core)          # (416, 1000, 3)
tucker_repr  = core_np.reshape(416, -1)   # (416, 3000)

print(f"Core shape         : {core_np.shape}")
print(f"Tucker repr shape  : {tucker_repr.shape}")
print(f"Non-zero var cols  : {(np.var(tucker_repr, axis=0) > 1e-10).sum()} / {tucker_repr.shape[1]}")
print(f"NaN values         : {np.isnan(tucker_repr).sum()}")

Tucker decomposition rank=(416, 1000, 3) ... done in 26.5s
Core shape         : (416, 1000, 3)
Tucker repr shape  : (416, 3000)
Non-zero var cols  : 3000 / 3000
NaN values         : 0


## Step 3 — PCA at 90%

Compresses the (416, 3,000) Tucker representation to ~57 final components.  
**Why 90% and not 70-80%:** one dominant direction captures 70-80% of Tucker variance
(overall brain shape, non-dementia-specific). The remaining 10% variance contains the
subtle, systematic differences relevant to dementia. 90% PCA forces inclusion of those directions.

In [5]:
PCA_THRESH = 0.90
SAFE_LIMIT = len(patient_ids) / 3   # 138.67

pca       = PCA(n_components=PCA_THRESH, random_state=42)
X_final   = pca.fit_transform(tucker_repr)
N_FINAL   = X_final.shape[1]
var_total = pca.explained_variance_ratio_.sum() * 100

print(f"Tucker repr  : {tucker_repr.shape}")
print(f"After PCA    : {X_final.shape}  ({var_total:.1f}% of Tucker variance)")
print(f"Safe limit   : < {SAFE_LIMIT:.2f}")
print(f"Status       : {'SAFE ✓' if N_FINAL < SAFE_LIMIT else 'UNSAFE ✗'}  ({N_FINAL} components)")

Tucker repr  : (416, 3000)
After PCA    : (416, 57)  (90.0% of Tucker variance)
Safe limit   : < 138.67
Status       : SAFE ✓  (57 components)


## Step 4 — Robust Mahalanobis distance matrix

In [6]:
print(f"Computing Robust Mahalanobis on X_final {X_final.shape} ...", end=" ", flush=True)
S_est = S_robust(X_final, method="trimmed", alpha=0.05)
D     = robust_mahalanobis_dist_matrix(X_final, S_est)
print("done")
print(f"D shape      : {D.shape}")
print(f"Value range  : [{D.min():.4f}, {D.max():.4f}]")
print(f"NaN values   : {np.isnan(D).sum()}")
print(f"Is symmetric : {np.allclose(D, D.T)}")

Computing Robust Mahalanobis on X_final (416, 57) ... done
D shape      : (416, 416)
Value range  : [0.0000, 35.9011]
NaN values   : 0
Is symmetric : True


## Step 5 — Cluster and evaluate (k=3 and k=4)

Same evaluation as notebook 13: spread · silhouette · ARI · NMI.

In [7]:
all_results = []

print(f"{'='*65}")
print(f"  Tucker rank={TUCKER_RANK}  PCA={PCA_THRESH*100:.0f}%  "
      f"X_final={X_final.shape}  CDR patients={mask_cdr.sum()}")
print(f"{'='*65}")

for k in [3, 4]:
    result = kmedoids.fasterpam(D, medoids=k, random_state=42)
    labels = np.array(result.labels, dtype=int)

    # ── % healthy per cluster ──────────────────────────────────────────────────
    df_eval = pd.DataFrame({"cluster": labels, "CDR": cdr_series.values})
    df_eval = df_eval[df_eval["CDR"].notnull()].copy()
    pct_tab = pd.crosstab(
        df_eval["cluster"], df_eval["CDR"], normalize="index"
    ).mul(100).round(1)
    pct_h  = pct_tab[0.0] if 0.0 in pct_tab.columns \
             else pd.Series(0.0, index=pct_tab.index)
    spread = float(pct_h.max() - pct_h.min())

    # ── CDR crosstab ───────────────────────────────────────────────────────────
    df_ev2 = df_eval.copy()
    df_ev2["CDR"] = df_ev2["CDR"].astype(str)
    ct = pd.crosstab(df_ev2["cluster"], df_ev2["CDR"],
                     margins=True, margins_name="Total")
    print(f"\n  k={k}  loss={result.loss:.4f}  "
          f"sizes={ {c: int((labels==c).sum()) for c in sorted(set(labels))} }")
    print("  Count:")
    print(ct.to_string())
    print("  Row %:")
    print(pct_tab.to_string())

    # ── Metrics ────────────────────────────────────────────────────────────────
    sil = silhouette_score(D, labels, metric="precomputed")
    ari = adjusted_rand_score(cdr_int, labels[mask_cdr])
    nmi = normalized_mutual_info_score(cdr_int, labels[mask_cdr])

    print(f"\n  spread={spread:.2f}%  sil={sil:.4f}  ARI={ari:.4f}  NMI={nmi:.4f}")

    all_results.append({
        "method":             f"tucker_{TUCKER_RANK}_pca{PCA_THRESH*100:.0f}",
        "n_final_components": N_FINAL,
        "k":                  k,
        "silhouette":         round(sil, 4),
        "ari":                round(ari, 4),
        "nmi":                round(nmi, 4),
        "spread_pct_healthy": round(spread, 2),
    })

print("\n\nClustering complete.")

  Tucker rank=(416, 1000, 3)  PCA=90%  X_final=(416, 57)  CDR patients=235

  k=3  loss=1294.1249  sizes={np.int64(0): 1, np.int64(1): 414, np.int64(2): 1}
  Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0          1    0    0    0      1
1        134   70   28    2    234
Total    135   70   28    2    235
  Row %:
CDR        0.0   0.5   1.0  2.0
cluster                        
0        100.0   0.0   0.0  0.0
1         57.3  29.9  12.0  0.9

  spread=42.70%  sil=0.7794  ARI=-0.0043  NMI=0.0047

  k=4  loss=1270.5585  sizes={np.int64(0): 1, np.int64(1): 413, np.int64(2): 1, np.int64(3): 1}
  Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0          1    0    0    0      1
1        134   70   28    2    234
Total    135   70   28    2    235
  Row %:
CDR        0.0   0.5   1.0  2.0
cluster                        
0        100.0   0.0   0.0  0.0
1         57.3  29.9  12.0  0.9

  spread=42.70%  sil=0.7799  ARI=-0.0043  NMI=0.00

## Step 6 — Summary and save

In [8]:
df_results = pd.DataFrame(all_results)

print(f"Tucker rank={TUCKER_RANK}  PCA={PCA_THRESH*100:.0f}%  "
      f"final={N_FINAL} components")
print("=" * 70)
print(df_results.to_string(index=False))
print(f"\nSilhouette : all {len(patient_ids)} patients")
print(f"ARI / NMI  : {mask_cdr.sum()} patients with CDR  (CDR × 2)")

out_path = os.path.join(NB_PADDED, "results_tucker_clustering.csv")
df_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

Tucker rank=(416, 1000, 3)  PCA=90%  final=57 components
                     method  n_final_components  k  silhouette     ari    nmi  spread_pct_healthy
tucker_(416, 1000, 3)_pca90                  57  3      0.7794 -0.0043 0.0047                42.7
tucker_(416, 1000, 3)_pca90                  57  4      0.7799 -0.0043 0.0047                42.7

Silhouette : all 416 patients
ARI / NMI  : 235 patients with CDR  (CDR × 2)

Saved: C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\new_results_with_padding\results_tucker_clustering.csv
